# 07_analysis — Framing shift analysis

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `Theory/README.md` → Environment setup.

**Input:** `data/output/networks/cooccurrence/` + `actor_actor/` (GraphML)
**Output:** tables and figures under `data/output/analysis/<level>/`

## What this notebook computes, by level

| Level | Output | Weighted? |
|---|---|---|
| **Dyad** | `dyad_level/edge_weight_timeseries.png`, `actor_concept_centrality.png` | yes |
| **Graph** | `graph_level/graph_level_metrics.csv` | structural |
| **Node** | `node_level/node_katz_trajectories.png` (headline) + `coverage_diagnostics.png`, `centrality_resolution.csv` | Katz yes, rest no |
| **Group** | `group_level/group_volume_share.*`, `group_concept_share.*` | yes |
| **Projection** | `projection/projection_metrics.csv`, `projection_top_pairs.csv` | derived view / negative result |
| **Polarity** | `polarity/polarity_over_time.*` | volume yes, breadth no |

**Reading rule for this notebook.** The bipartite graph saturates as the crisis
peaks (density 0.59 → 0.93), so *unweighted* measures collapse toward constants
and describe the design rather than the discourse. Weighted measures — dyad edge
weights, Katz, group volume share, polarity volume share — carry the signal.
Unweighted ones are reported as **saturation diagnostics**, never as rankings.

## Visual identity

**Teal** for buildup/aftermath windows, **purple** for climax; climax windows are
shaded purple on every time-axis figure.

## Pipeline steps in this notebook

1. Setup & paths
2. Load networks (GraphML) + edge files
3. Key dyads + visual style + window ordering
4. Edge weight time series for key dyads
5. Concept centrality per actor over time
6. ~~Source-level comparison~~ (disabled — superseded by `compare_sources.ipynb`)
7. Graph-level metrics (bipartite)
8. Node-level centralities — Katz headline + coverage diagnostics
9. Group-level framing volume (coalition share of voice)
10. Projection analysis (actor congruence network)
11. Framing polarity over time (volume vs. breadth)
12. Validation checkpoint

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.alias_map    import ACTOR_WHITELIST
from src.concept_dict import CONCEPT_DICT, CONCEPT_POLARITY, POLARITY_COLOR
from src.time_windows import TIME_WINDOWS, window_file_label
from src.actor_groups import ACTOR_ALIGNMENT, GROUP_ORDER

OUTPUT_DIR   = ROOT / 'data' / 'output'
NETWORKS_DIR = OUTPUT_DIR / 'networks'
EDGES_DIR    = OUTPUT_DIR / 'edges'
ANALYSIS_DIR = OUTPUT_DIR / 'analysis'
# Outputs are grouped by LEVEL OF ANALYSIS (each folder holds its own tables
# and figures together).
DIR_DYAD        = ANALYSIS_DIR / 'dyad_level'
DIR_GRAPH       = ANALYSIS_DIR / 'graph_level'
DIR_NODE        = ANALYSIS_DIR / 'node_level'
DIR_GROUP       = ANALYSIS_DIR / 'group_level'
DIR_PROJECTION  = ANALYSIS_DIR / 'projection'
DIR_POLARITY    = ANALYSIS_DIR / 'polarity'
for _d in (DIR_DYAD, DIR_GRAPH, DIR_NODE, DIR_GROUP,
           DIR_PROJECTION, DIR_POLARITY):
    _d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Networks dir : {NETWORKS_DIR}')
print(f'Edges dir    : {EDGES_DIR}')
print(f'Analysis dir : {ANALYSIS_DIR}')
assert NETWORKS_DIR.exists(), 'ERROR: networks dir missing — run 06_networks first'


## Step 2: Load networks and edge files

Two read paths:
- **GraphML** files give us the full bipartite graph per window, including isolated whitelist nodes and node-level stats (`degree`, `total_weight`, `total_weight_norm`).
- **JSONL edge files** carry the full `sources` dict per edge — GraphML cannot serialize Python dicts on edges, so notebook 06 collapses sources to `top_source` / `n_sources`. We need the full breakdown for Step 6.

In [ ]:
# All windows in chronological order (from src/time_windows.py)
WINDOW_ORDER = [w[0] for w in TIME_WINDOWS]

# Load networks
graphs = {}
for window in WINDOW_ORDER:
    path = NETWORKS_DIR / 'cooccurrence' / 'graphml' / f'{window_file_label(window)}_cooc.graphml'
    if path.exists():
        graphs[window] = nx.read_graphml(path)

# Coerce GraphML string attributes back to numeric for edges
for G in graphs.values():
    for _, _, d in G.edges(data=True):
        d['weight']            = int(float(d['weight']))
        d['weight_normalized'] = float(d['weight_normalized'])
    for _, d in G.nodes(data=True):
        if 'total_weight' in d:
            d['total_weight'] = int(float(d['total_weight']))
        if 'total_weight_norm' in d:
            d['total_weight_norm'] = float(d['total_weight_norm'])

# Load full edge JSONL records (source breakdown is preserved here)
edges_by_window = defaultdict(list)
for window in WINDOW_ORDER:
    f = EDGES_DIR / f'edges_{window}.jsonl'
    if not f.exists():
        continue
    with open(f, encoding='utf-8') as fh:
        for line in fh:
            edges_by_window[window].append(json.loads(line))

present_windows = [w for w in WINDOW_ORDER if w in graphs]
missing_windows = [w for w in WINDOW_ORDER if w not in graphs]

print(f'Loaded {len(graphs)} networks: {present_windows}')
if missing_windows:
    print(f'Missing (no data yet)       : {missing_windows}')
    print('  → analyses below render only windows present in the current corpus')

assert present_windows, 'No GraphML files loaded — run 06_networks first'

# Actor–actor projections (06 Step 8) — for the projection-level analysis
projections = {}
for window in present_windows:
    pth = NETWORKS_DIR / 'actor_actor' / 'graphml' / f'{window_file_label(window)}_actors.graphml'
    if pth.exists():
        Gp = nx.read_graphml(pth)
        for _, _, d in Gp.edges(data=True):
            d['weight'] = int(float(d['weight']))
            for k in ('n_shared_concepts', 'polarity_positive', 'polarity_neutral', 'polarity_negative'):
                if k in d:
                    d[k] = int(float(d[k]))
        projections[window] = Gp
print(f'Loaded {len(projections)} actor–actor projections')

## Step 3: Key dyads + visual style

The five key dyads from README §6 trace the central narrative arcs of the thesis:

| Dyad | Expected pattern |
|---|---|
| (USA, military_action) | peak Jun 22–24 |
| (IRAN, deterrence)     | shift from buildup to aftermath |
| (IAEA, nuclear_program)| persistent; watch for valence shift |
| (USA, strike_claims)      | spike Jun 22, then decay |
| (IRAN, diplomacy)      | rise in aftermath |

Color scheme follows README: **teal `#1f8fa6`** for buildup/aftermath windows, **purple `#7e57c2`** for climax. Concept colors use a colorblind-safe qualitative palette.

In [ ]:
KEY_DYADS = [
    ('USA',  'military_action'),
    ('IRAN', 'deterrence'),
    ('IAEA', 'nuclear_program'),
    ('USA',  'strike_claims'),
    ('IRAN', 'diplomacy'),
]

# Phase palette (README §6 visual identity)
TEAL    = '#1f8fa6'
PURPLE  = '#7e57c2'

def phase_color(window: str) -> str:
    """Teal for buildup/aftermath, purple for climax."""
    return PURPLE if window.startswith('climax') else TEAL

# Concept palette
CONCEPT_COLOR = {
    'military_action': '#d62728',  # red
    'nuclear_program': '#ff7f0e',  # orange
    'deterrence':      '#1f77b4',  # blue
    'diplomacy':       '#2ca02c',  # green
    'strike_claims':      '#9467bd',  # purple
}

# Dyad colors mirror the concept of each dyad for visual coherence
DYAD_COLOR = {
    ('USA',  'military_action'): CONCEPT_COLOR['military_action'],
    ('IRAN', 'deterrence'):      CONCEPT_COLOR['deterrence'],
    ('IAEA', 'nuclear_program'): CONCEPT_COLOR['nuclear_program'],
    ('USA',  'strike_claims'):      CONCEPT_COLOR['strike_claims'],
    ('IRAN', 'diplomacy'):       CONCEPT_COLOR['diplomacy'],
}

def shade_climax(ax, windows):
    """Add a translucent purple band over climax windows on the x-axis."""
    for i, w in enumerate(windows):
        if w.startswith('climax'):
            ax.axvspan(i - 0.5, i + 0.5, color=PURPLE, alpha=0.10, zorder=0)


## Step 4: Edge weight time series for key dyads

Per README §6 figure 1. **Normalized** weight (mentions per article) is plotted because raw weights cannot be compared across windows with different article counts.

Climax windows are shaded purple. When the corpus extends through climax/aftermath, this is the figure that should reveal the canonical Midnight Hammer arc: USA–military_action and USA–strike_claims peak in climax_w2, IRAN–diplomacy rises through aftermath, IRAN–deterrence shifts from buildup to aftermath.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))
shade_climax(ax, present_windows)

x = np.arange(len(present_windows))
for actor, concept in KEY_DYADS:
    weights = []
    for window in present_windows:
        G = graphs[window]
        if G.has_edge(actor, concept):
            weights.append(G[actor][concept]['weight_normalized'])
        else:
            weights.append(0.0)
    ax.plot(
        x, weights,
        marker='o', linewidth=2.2, markersize=7,
        color=DYAD_COLOR[(actor, concept)],
        label=f'{actor} — {concept}',
    )

ax.set_xticks(x)
ax.set_xticklabels(present_windows, rotation=30, ha='right')
ax.set_xlabel('Time window')
ax.set_ylabel('Normalized edge weight (mentions per article)')
ax.set_title('Edge weight time series — key actor–concept dyads')
ax.grid(axis='y', alpha=0.3)

# Stack the dyad legend with a 'climax window' shading chip
climax_patch = mpatches.Patch(color=PURPLE, alpha=0.20, label='climax window')
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + [climax_patch], loc='upper right', framealpha=0.95)

plt.tight_layout()
out = DIR_DYAD / 'edge_weight_timeseries.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')


## Step 5: Concept centrality per actor over time

For each top actor (ranked by total normalized weight across all windows), we ask: **which concept has the highest edge weight in each window?** The answer is plotted as a categorical heatmap — actors on rows, windows on columns, cell color encoding the dominant concept.

"—" cells mark windows where the actor has no edges at all.

A text table prints the same data for the thesis appendix.

In [ ]:
TOP_ACTORS_N = 12

# Rank actors by total normalized weight across all present windows
actor_total_norm = Counter()
for window in present_windows:
    G = graphs[window]
    for actor in [n for n, d in G.nodes(data=True) if d['node_type'] == 'actor']:
        actor_total_norm[actor] += float(G.nodes[actor].get('total_weight_norm', 0.0))

top_actors = [a for a, w in actor_total_norm.most_common() if w > 0][:TOP_ACTORS_N]
print(f'Top {len(top_actors)} actors by total normalized weight:')
for a in top_actors:
    print(f'  {actor_total_norm[a]:7.3f}  {a}')

# Build top-concept matrix
concepts = list(CONCEPT_DICT.keys())
concept_to_idx = {c: i for i, c in enumerate(concepts)}

NA = -1
top_concept_matrix = np.full((len(top_actors), len(present_windows)), NA, dtype=int)

for i, actor in enumerate(top_actors):
    for j, window in enumerate(present_windows):
        G = graphs[window]
        if actor not in G:
            continue
        best, best_w = None, -1.0
        for _, neighbor, d in G.edges(actor, data=True):
            if neighbor in concept_to_idx and d['weight_normalized'] > best_w:
                best, best_w = neighbor, d['weight_normalized']
        if best is not None:
            top_concept_matrix[i, j] = concept_to_idx[best]

# Plot
cmap_colors = [CONCEPT_COLOR[c] for c in concepts]
cmap = ListedColormap(cmap_colors)

fig, ax = plt.subplots(
    figsize=(max(8, 1.4 * len(present_windows) + 3), 0.55 * len(top_actors) + 2)
)
display = np.ma.masked_where(top_concept_matrix == NA, top_concept_matrix)
ax.imshow(display, cmap=cmap, aspect='auto', vmin=0, vmax=len(concepts) - 1)

ax.set_xticks(range(len(present_windows)))
ax.set_xticklabels(present_windows, rotation=30, ha='right')
ax.set_yticks(range(len(top_actors)))
ax.set_yticklabels(top_actors)
ax.set_title('Top concept per actor per window (categorical centrality)')

SHORT = {'military_action': 'mil', 'nuclear_program': 'nuke',
         'deterrence': 'det', 'diplomacy': 'dip', 'strike_claims': 'stc'}
for i in range(len(top_actors)):
    for j in range(len(present_windows)):
        idx = top_concept_matrix[i, j]
        if idx == NA:
            ax.text(j, i, '—', ha='center', va='center',
                    color='lightgray', fontsize=9)
        else:
            ax.text(j, i, SHORT[concepts[idx]], ha='center', va='center',
                    color='white', fontsize=9, weight='bold')

handles = [mpatches.Patch(color=CONCEPT_COLOR[c], label=c) for c in concepts]
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
out = DIR_DYAD / 'actor_concept_centrality.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

# Text table for appendix
print('\nText summary (top concept per actor per window):')
col_w = 11
header = f'{"actor":<12s} | ' + ' | '.join(f'{w[:col_w]:<{col_w}s}' for w in present_windows)
print(header)
print('-' * len(header))
for i, a in enumerate(top_actors):
    row = []
    for j in range(len(present_windows)):
        idx = top_concept_matrix[i, j]
        row.append('—' if idx == NA else concepts[idx][:col_w])
    print(f'{a:<12s} | ' + ' | '.join(f'{r:<{col_w}s}' for r in row))


## Step 6: Source-level comparison (DISABLED)

Per README §6, this analysis splits edge weight by source tier (US elite / US regional / Non-Western) to test whether framings differ across outlet types — most useful when the corpus mixes US and international wires.

**This step is currently disabled** because the present corpus is sourced exclusively from North America, so the cross-tier comparison would be trivially one-sided. The code is preserved (commented out) in the next cell and can be re-enabled at any time once non-Western or international sources are added to `data/raw/`.

In [ ]:
# =============================================================================
# Source-level comparison is DISABLED.
#
# The current corpus is exclusively North American, so the US-elite vs.
# US-regional vs. Non-Western tier split would be trivially one-sided. Code is
# preserved below so the analysis can be re-enabled once international sources
# (Al Jazeera, Reuters int'l, AFP, etc.) are added to data/raw/ and re-parsed.
#
# To re-enable: uncomment the block below.
# =============================================================================

# US_ELITE = {
#     'The New York Times', 'The Washington Post', 'The Wall Street Journal',
#     'USA Today', 'USA Today Online', 'The Christian Science Monitor',
#     'Bloomberg', 'Time Magazine', 'Time',
#     'Reuters - U.S.', 'AP', 'The Associated Press',
# }
# NON_WESTERN = {
#     'Al Jazeera', 'Reuters', 'AFP', 'ASEAN Tribune',
#     'Xinhua', 'TASS', 'IRNA', 'Press TV', 'Tehran Times',
#     'South China Morning Post', 'The Guardian',
# }
#
# def classify_source(src: str) -> str:
#     if not src:
#         return 'US regional / other'
#     if src in US_ELITE:
#         return 'US elite'
#     if src in NON_WESTERN:
#         return 'Non-Western / wire'
#     return 'US regional / other'
#
# TIERS = ['US elite', 'US regional / other', 'Non-Western / wire']
# TIER_COLOR = {
#     'US elite':              '#1f77b4',
#     'US regional / other':   '#aec7e8',
#     'Non-Western / wire':    '#ff7f0e',
# }
#
# # Build (dyad × window × tier) weight matrix from edge JSONL records
# matrix = {dyad: {w: Counter() for w in present_windows} for dyad in KEY_DYADS}
# for window in present_windows:
#     for edge in edges_by_window.get(window, []):
#         dyad = (edge['actor'], edge['concept'])
#         if dyad not in matrix:
#             continue
#         for src, count in edge.get('sources', {}).items():
#             matrix[dyad][window][classify_source(src)] += count
#
# # Plot — one panel per dyad, stacked bars per window
# fig, axes = plt.subplots(
#     len(KEY_DYADS), 1,
#     figsize=(11, 2.0 * len(KEY_DYADS) + 1),
#     sharex=True,
# )
# if len(KEY_DYADS) == 1:
#     axes = [axes]
#
# x = np.arange(len(present_windows))
# width = 0.7
#
# for ax, dyad in zip(axes, KEY_DYADS):
#     shade_climax(ax, present_windows)
#     bottoms = np.zeros(len(present_windows))
#     for tier in TIERS:
#         heights = np.array(
#             [matrix[dyad][w].get(tier, 0) for w in present_windows], dtype=float
#         )
#         ax.bar(x, heights, width, bottom=bottoms, color=TIER_COLOR[tier], label=tier)
#         bottoms += heights
#     ax.set_title(f'{dyad[0]} — {dyad[1]}', fontsize=10, loc='left')
#     ax.set_ylabel('mentions')
#     ax.grid(axis='y', alpha=0.3)
#
# axes[0].legend(loc='upper right', fontsize=9, framealpha=0.95)
# axes[-1].set_xticks(x)
# axes[-1].set_xticklabels(present_windows, rotation=30, ha='right')
# fig.suptitle('Source-tier contributions to key dyads, by window', fontsize=12)
#
# plt.tight_layout()
# out = FIGURES_DIR / 'source_comparison.png'
# plt.savefig(out, dpi=150, bbox_inches='tight')
# plt.show()
# print(f'Saved: {out}')
#
# # Tier breakdown table for the thesis writeup
# print('\nTier totals across all present windows:')
# for dyad in KEY_DYADS:
#     total_per_tier = Counter()
#     for w in present_windows:
#         total_per_tier.update(matrix[dyad][w])
#     total = sum(total_per_tier.values()) or 1
#     parts = [
#         f'{tier}: {total_per_tier[tier]} ({100 * total_per_tier[tier] / total:.0f}%)'
#         for tier in TIERS
#     ]
#     print(f'  {dyad[0]:>6s} — {dyad[1]:<16s} | {"   ".join(parts)}')

print('Source-level comparison is disabled (corpus is exclusively North American).')


## Step 7: Graph-level metrics (bipartite)

One row per window — the top-level descriptive table for the thesis.

Standard `nx.transitivity` and `nx.average_clustering` are **0 by construction** on a bipartite graph: a triangle needs three mutually-adjacent nodes, impossible across two disjoint sets. So we use the **Latapy et al. bipartite clustering** (`bipartite.clustering`) here, and compute standard clustering/transitivity on the actor–actor **projection** instead (Step 11). Likewise **reciprocity is undefined for undirected graphs**, so it is reported only on the directed SVO network (Step 13).

In [ ]:
graph_rows = []
for window in present_windows:
    G = graphs[window]
    actors   = {n for n, d in G.nodes(data=True) if d['node_type'] == 'actor'}
    concepts = {n for n, d in G.nodes(data=True) if d['node_type'] == 'concept'}
    clustering = bipartite.clustering(G)
    graph_rows.append({
        'window':               window,
        'density':              bipartite.density(G, actors),
        'bipartite_clustering': float(np.mean(list(clustering.values()))) if clustering else 0.0,
        'n_actor_nodes':        len(actors),
        'n_concept_nodes':      len(concepts),
        'n_edges':              G.number_of_edges(),
        'total_edge_weight':    int(sum(d['weight'] for _, _, d in G.edges(data=True))),
    })

graph_df = pd.DataFrame(graph_rows).set_index('window')
graph_df.to_csv(DIR_GRAPH / 'graph_level_metrics.csv')
print('Saved: graph_level_metrics.csv')
print(graph_df.round(4).to_string())

## Step 8: Node-level centralities (per actor, per window)

**Katz is the headline metric; the other three are saturation diagnostics.**

Long format: `window, actor, metric, value, centrality_method`. Degree,
betweenness and closeness use the **bipartite-normalised** NetworkX functions —
but they are *unweighted*, and with only five concepts they barely discriminate:
in `climax_w2` they sort 43 actors into just 3–4 buckets and induce the
**identical** 2/5/36 partition, i.e. they carry one metric's worth of
information between them. They are reported as **coverage diagnostics**, never
as rankings, and `centrality_resolution.csv` quantifies exactly how coarse they
are (distinct values per metric per window).

The weighted metric is labelled **`eigenvector_katz`** — deliberately, because
no eigenvector centrality is ever computed here: a bipartite adjacency has a
symmetric ±λ spectrum, so power iteration oscillates and never converges (it
fails on *every* window). The notebook falls back to weighted **Katz**
(`katz_centrality_numpy`, spectrum-safe α = 0.85/λ_max) and records which method
produced each row. Because it reads edge weights, it is the only node-level
measure that genuinely ranks actors (38–42 distinct values across 43 actors).

**Degree vs Katz:** degree = *how many* concepts an actor touches; Katz = *how
important* those attachments are. High-degree/low-Katz = visible but peripheral;
low-degree/high-Katz = selectively central.

In [ ]:
def eigen_or_katz(G):
    """Weighted eigenvector centrality; falls back to spectrum-safe Katz when the
    bipartite power iteration fails to converge (oscillation on a +/-lambda spectrum)."""
    try:
        return nx.eigenvector_centrality(G, weight='weight', max_iter=1000), 'eigenvector'
    except nx.PowerIterationFailedConvergence:
        A = nx.to_numpy_array(G, weight='weight')
        lam = max(abs(np.linalg.eigvals(A))) if A.size else 1.0
        alpha = 0.85 / lam if lam > 0 else 0.1
        return nx.katz_centrality_numpy(G, alpha=alpha, weight='weight'), 'katz_fallback'

KATZ            = 'eigenvector_katz'          # weighted -> the headline metric
DIAGNOSTIC_MTRS = ['degree_centrality', 'betweenness_centrality', 'closeness_centrality']
metric_list     = DIAGNOSTIC_MTRS + [KATZ]

node_rows, fallback_windows = [], []
for window in present_windows:
    G = graphs[window]
    actors = {n for n, d in G.nodes(data=True) if d['node_type'] == 'actor'}
    cmaps = {
        'degree_centrality':      bipartite.degree_centrality(G, actors),
        'betweenness_centrality': bipartite.betweenness_centrality(G, actors),
        'closeness_centrality':   bipartite.closeness_centrality(G, actors),
    }
    eig, method = eigen_or_katz(G)
    cmaps[KATZ] = eig
    if method == 'katz_fallback':
        fallback_windows.append(window)
    for metric in metric_list:
        m = method if metric == KATZ else 'exact'
        for actor in actors:
            node_rows.append({'window': window, 'actor': actor, 'metric': metric,
                              'value': float(cmaps[metric].get(actor, 0.0)),
                              'centrality_method': m})

node_df = pd.DataFrame(node_rows)
node_df.to_csv(DIR_NODE / 'node_level_centralities.csv', index=False)
print(f'Saved: node_level_centralities.csv  ({len(node_df)} rows)')
if fallback_windows:
    print(f'Eigenvector -> Katz fallback fired in {len(fallback_windows)}/{len(present_windows)} '
          f'window(s) — bipartite spectra never converge under power iteration.')

# --- resolution diagnostic: how many DISTINCT values does each metric produce? --
res = (node_df.groupby(['window', 'metric']).value.nunique()
       .unstack().reindex(present_windows)[metric_list])
# Weighted degree (strength) is not one of the reported centralities, but its
# resolution is the counterfactual the node-level negative result rests on: it
# shows the collapse comes from discarding the weights, not from the data.
res['weighted_degree'] = [
    len({round(sum(d['weight'] for _, _, d in graphs[w].edges(a, data=True)), 6)
         for a, nd in graphs[w].nodes(data=True) if nd['node_type'] == 'actor'})
    for w in present_windows
]
# In the UNWEIGHTED graph an actor's whole structural position is the subset of
# concept clusters it attaches to, so two actors with the same subset are
# automorphic and no structure-only metric can separate them. The count of
# occupied subsets is therefore a hard upper bound on the resolution of degree,
# betweenness and closeness alike. With |C| concepts it cannot exceed 2**|C|.
res['occupied_concept_subsets'] = [
    len({frozenset(graphs[w].neighbors(a))
         for a, nd in graphs[w].nodes(data=True) if nd['node_type'] == 'actor'})
    for w in present_windows
]
res.to_csv(DIR_NODE / 'centrality_resolution.csv')
print('\nSaved: centrality_resolution.csv  — distinct values per metric per window')
print(f'(out of {node_df.actor.nunique()} actors; 1 = useless, n = perfect discrimination)')
print(res.to_string())

# --- FIGURE 1 (headline): Katz trajectories ------------------------------------
katz = node_df[node_df.metric == KATZ]
piv = katz.pivot_table(index='actor', columns='window', values='value').reindex(columns=present_windows)
top_actors_katz = piv.mean(axis=1).sort_values(ascending=False).head(8).index.tolist()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(present_windows))
for actor in top_actors_katz:
    ax.plot(x, piv.loc[actor, present_windows].values, marker='o', linewidth=2.2,
            markersize=6, label=actor)
shade_climax(ax, present_windows)
ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=30, ha='right')
ax.set_ylabel('Katz centrality (weighted)')
ax.set_title('Node level — Katz centrality trajectories (the only weighted node metric)')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
out = DIR_NODE / 'node_katz_trajectories.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'\nSaved: {out.name}')

# --- FIGURE 2 (diagnostic): the three unweighted metrics collapsing -------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, DIAGNOSTIC_MTRS):
    p = (node_df[node_df.metric == metric]
         .pivot_table(index='actor', columns='window', values='value')
         .reindex(columns=present_windows))
    for actor in p.mean(axis=1).sort_values(ascending=False).head(6).index:
        ax.plot(x, p.loc[actor, present_windows].values, marker='o', linewidth=1.6, markersize=4)
    shade_climax(ax, present_windows)
    ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=45, ha='right', fontsize=7)
    ax.set_title(f'{metric.replace("_", " ")}\n({res[metric].min()}-{res[metric].max()} distinct values)',
                 fontsize=9)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Coverage diagnostics — unweighted centralities (NOT rankings; they collapse as the graph saturates)',
             fontsize=11)
plt.tight_layout()
out = DIR_NODE / 'coverage_diagnostics.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')

## Step 9: Group-level framing volume (coalition share of voice)

Aggregating the *mean of node centralities* by coalition does not work here: over
half of every group's members sit at the Katz floor, so the mean mostly reports
**how many inactive actors a group contains**, not how the coalition is framed
(the old series was flat at 0.116–0.128 across the entire event, and adding
members would only dilute it further).

We therefore aggregate **framing volume** instead — the summed edge weight of a
coalition's actors, as a share of all mentions in that window:

```
volume_share(g, t) = Σ w(a, c, t) for a ∈ g   ÷   Σ w(a, c, t) over all actors
```

Inactive actors contribute 0 to both numerator and denominator, so they cannot
drag the measure down. It sits on a natural scale (shares summing to 1), moves
with the discourse rather than with membership choices, and — because it is a
share rather than a structural score — it is **directly comparable between the
Western and Iranian corpora**.

The same quantity split by concept gives the coalition × framing view: for each
concept we report both `share_of_concept` (which coalition owns this framing) and
`share_of_group` (what this coalition talks about). Group membership comes from
`ACTOR_ALIGNMENT` in `src/actor_groups.py`, which is audited for exhaustive
coverage of `ACTOR_WHITELIST` at import time.

In [ ]:
from collections import Counter as _Counter

grp_rows, gc_rows = [], []
for window in present_windows:
    G = graphs[window]
    by_g, by_gc, by_c = _Counter(), _Counter(), _Counter()
    total = 0
    for u, v, d in G.edges(data=True):
        a, c = (u, v) if G.nodes[u]['node_type'] == 'actor' else (v, u)
        g = ACTOR_ALIGNMENT.get(a, 'unassigned')
        w = d['weight']
        by_g[g] += w; by_gc[(g, c)] += w; by_c[c] += w; total += w
    total = total or 1
    for g in GROUP_ORDER:
        grp_rows.append({'window': window, 'group': g,
                         'volume': by_g.get(g, 0),
                         'volume_share': by_g.get(g, 0) / total})
        for c in CONCEPT_DICT:
            v = by_gc.get((g, c), 0)
            gc_rows.append({'window': window, 'group': g, 'concept': c, 'volume': v,
                            'share_of_concept': v / (by_c.get(c, 0) or 1),
                            'share_of_group':   v / (by_g.get(g, 0) or 1)})

group_df = pd.DataFrame(grp_rows)
gc_df    = pd.DataFrame(gc_rows)
group_df.to_csv(DIR_GROUP / 'group_volume_share.csv', index=False)
gc_df.to_csv(DIR_GROUP / 'group_concept_share.csv', index=False)
print(f'Saved: group_volume_share.csv ({len(group_df)} rows), '
      f'group_concept_share.csv ({len(gc_df)} rows)')

share = group_df.pivot(index='window', columns='group', values='volume_share').reindex(present_windows)
print('\nCoalition share of framing volume per window:')
print(share[GROUP_ORDER].round(3).to_string())

GROUP_COLOR = {'us_aligned': '#1f77b4', 'iran_aligned': '#d62728',
               'neutral': '#2ca02c', 'other': '#7f7f7f'}

# --- FIGURE 1: coalition share of voice over time ------------------------------
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(present_windows))
for g in GROUP_ORDER:
    ax.plot(x, share[g].values, marker='o', linewidth=2.4, markersize=7,
            color=GROUP_COLOR[g], label=g)
shade_climax(ax, present_windows)
ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=30, ha='right')
ax.set_ylabel('Share of framing volume (mentions)')
ax.set_title('Group level — coalition share of framing volume over time')
ax.grid(axis='y', alpha=0.3); ax.legend(loc='upper right')
plt.tight_layout()
out = DIR_GROUP / 'group_volume_share.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')

# --- FIGURE 2: who owns each framing (share_of_concept), one panel per concept --
concepts = list(CONCEPT_DICT)
fig, axes = plt.subplots(1, len(concepts), figsize=(4 * len(concepts), 4), sharey=True)
for ax, c in zip(np.atleast_1d(axes), concepts):
    sub = gc_df[gc_df.concept == c].pivot(index='window', columns='group',
                                          values='share_of_concept').reindex(present_windows)
    for g in GROUP_ORDER:
        ax.plot(x, sub[g].values, marker='o', linewidth=2, markersize=5,
                color=GROUP_COLOR[g], label=g)
    shade_climax(ax, present_windows)
    ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=45, ha='right', fontsize=7)
    ax.set_title(c, fontsize=10, color=CONCEPT_COLOR.get(c, '#333'))
    ax.grid(axis='y', alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel('Share of that concept\'s mentions')
np.atleast_1d(axes)[-1].legend(fontsize=8, loc='upper right')
fig.suptitle('Group level — which coalition owns each framing dimension', fontsize=12)
plt.tight_layout()
out = DIR_GROUP / 'group_concept_share.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')

## Step 10: Projection analysis — actor congruence network (Leifeld & Haunss 2012)

The actor–actor projections (notebook 06) are unipartite, so **standard** clustering/transitivity *are* meaningful here. We track `nx.density` (over connected actors), weighted `nx.average_clustering` and `nx.transitivity` over time, plus the **Latapy redundancy coefficient** of the concept nodes — the fraction of actor pairs sharing a concept that also share another one. We also list the strongest co-framing dyads (top-5 by weight, with their `dominant_polarity`). The projection is a **derived, secondary view**: with five concepts it is near-complete (density → 1, redundancy → 1), which is reported as a negative result.

In [ ]:
from networkx.algorithms import bipartite as _bip

proj_rows, pair_rows = [], []
for window in present_windows:
    P = projections.get(window)
    if P is None or P.number_of_edges() == 0:
        continue
    # Latapy redundancy coefficient on the CONCEPT nodes: the fraction of actor
    # pairs sharing a concept that also share another one. rc -> 1 means the
    # concept contributes no unique structure to the projection (saturation).
    G = graphs[window]
    Gc = G.subgraph([n for n in G if G.degree(n) > 0])
    concepts = [n for n, d in Gc.nodes(data=True) if d['node_type'] == 'concept']
    rc = _bip.node_redundancy(Gc, concepts) if len(concepts) > 1 else {}
    # Density over CONNECTED actors only: an actor with no concept edge has no
    # projection edge either, so including isolates deflates the saturation we
    # are measuring. n_actors_connected is reported alongside for transparency.
    Pc = P.subgraph([n for n in P if P.degree(n) > 0])
    proj_rows.append({'window': window,
                      'n_actors_connected': Pc.number_of_nodes(),
                      'density':        nx.density(Pc),
                      'avg_clustering': nx.average_clustering(P, weight='weight'),
                      'transitivity':   nx.transitivity(P),
                      'n_edges':        P.number_of_edges(),
                      'mean_redundancy_concepts': (sum(rc.values()) / len(rc)) if rc else float('nan')})
    for u, v, d in sorted(P.edges(data=True), key=lambda x: -x[2]['weight'])[:5]:
        pair_rows.append({'window': window, 'actor_a': u, 'actor_b': v,
                          'weight': d['weight'], 'dominant_polarity': d.get('dominant_polarity', '')})

proj_df  = pd.DataFrame(proj_rows).set_index('window')
pairs_df = pd.DataFrame(pair_rows)
proj_df.to_csv(DIR_PROJECTION / 'projection_metrics.csv')
pairs_df.to_csv(DIR_PROJECTION / 'projection_top_pairs.csv', index=False)
print('Saved: projection_metrics.csv, projection_top_pairs.csv')
print(proj_df.round(4).to_string())
print()
print('NOTE: with 5 concepts the projection is near-complete (density ~1, mean')
print('      concept redundancy ~1) — reported as a negative result. Actor-actor')
print('      structure is analysed on the directed SVO network (notebook 07b).')

## Step 11: Framing polarity over time (primary bipartite graph)

Computed on the **bipartite** `G_t` (not the lossy projection), in two variants
that must not be confused:

- **`volume_*` (primary)** — share of *mentions*: `Σ w on edges of that polarity ÷ Σ all w`.
- **`breadth_*` (secondary)** — share of *distinct links*, the previous definition.

The distinction is decisive. `CONCEPT_POLARITY` assigns 2 positive, 1 neutral and
2 negative clusters, so once the graph saturates (density → 0.93) the **breadth**
shares are pinned near the structural ratio 2/5 : 1/5 : 2/5 = 0.40 : 0.20 : 0.40
*regardless of what the press writes* — they report the dictionary's shape, not
the discourse. The volume shares are free to move, and they do.

Polarity is a property of the **concept invoked**, not sentence stance: there is
no negation detection, so an actor *rejecting* escalation still yields a
negative-cluster edge. Always describe these as shares of conflictual vs.
cooperative framing **vocabulary**, never as "positive/negative coverage".

In [ ]:
POLARITIES = ('positive', 'neutral', 'negative')

pol_rows = []
for window in present_windows:
    G = graphs[window]
    cnt, vol = Counter(), Counter()
    for _, _, d in G.edges(data=True):
        p = d.get('polarity', 'neutral')
        cnt[p] += 1
        vol[p] += d['weight']
    n_edges    = sum(cnt.values()) or 1
    n_mentions = sum(vol.values()) or 1
    row = {'window': window, 'n_edges': n_edges, 'n_mentions': n_mentions}
    for p in POLARITIES:
        row[f'volume_{p}']  = vol.get(p, 0) / n_mentions   # PRIMARY
        row[f'breadth_{p}'] = cnt.get(p, 0) / n_edges      # secondary
    pol_rows.append(row)

pol_df = pd.DataFrame(pol_rows).set_index('window')
pol_df.to_csv(DIR_POLARITY / 'polarity_over_time.csv')
print('Saved: polarity_over_time.csv')
print(pol_df[[f'volume_{p}' for p in POLARITIES] +
             [f'breadth_{p}' for p in POLARITIES]].round(3).to_string())

vol_rng = pol_df['volume_negative'].max() - pol_df['volume_negative'].min()
brd_rng = pol_df['breadth_negative'].max() - pol_df['breadth_negative'].min()
print(f'\nRange of the negative share across the event:')
print(f'  volume  (mentions) : {vol_rng:.3f}   <- the discourse signal')
print(f'  breadth (links)    : {brd_rng:.3f}   <- pinned near the 2:1:2 concept ratio')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
x = np.arange(len(present_windows))
for ax, kind, title in [
        (ax1, 'volume',  'PRIMARY — share of framing volume (mentions)'),
        (ax2, 'breadth', 'secondary — share of distinct links (saturation artefact)')]:
    for p in POLARITIES:
        ax.plot(x, pol_df.loc[present_windows, f'{kind}_{p}'].values, marker='o',
                linewidth=2.4, markersize=7, color=POLARITY_COLOR[p], label=p)
    shade_climax(ax, present_windows)
    ax.set_xticks(x); ax.set_xticklabels(present_windows, rotation=30, ha='right')
    ax.set_title(title, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
ax2.axhline(0.4, color='grey', ls=':', lw=1)
ax2.axhline(0.2, color='grey', ls=':', lw=1)
ax2.text(len(present_windows) - 0.6, 0.405, 'structural 2/5', fontsize=7, color='grey')
ax2.text(len(present_windows) - 0.6, 0.205, 'structural 1/5', fontsize=7, color='grey')
ax1.set_ylabel('Share'); ax1.legend(loc='upper left')
fig.suptitle('Framing polarity over time — conflictual vs. cooperative vocabulary', fontsize=12)
plt.tight_layout()
out = DIR_POLARITY / 'polarity_over_time.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}')

## Step 11b: Network figures for the thesis (construction + saturation)

In [ ]:
# ---------------------------------------------------------------------------
# Two figures the written thesis needs and the analysis above does not produce:
#   (a) how one sentence becomes edges, drawn from a real tagged sentence, and
#   (b) the bipartite graph itself, sparse window against saturated window.
# Both are generated from the data rather than drawn by hand, so they change
# with the corpus and cannot drift out of step with the numbers.
# ---------------------------------------------------------------------------
from matplotlib.patches import FancyBboxPatch

DIR_NETFIG = ANALYSIS_DIR / 'networks'
DIR_NETFIG.mkdir(parents=True, exist_ok=True)
INTERIM_DIR = ROOT / 'data' / 'interim'
TAGGED = INTERIM_DIR / 'sentences_tagged.jsonl'

POL_COLOR_FIG = {'positive': '#2e8b57', 'neutral': '#888888', 'negative': '#c0392b'}
ALIGN_COLOR = {'us_aligned': '#1f8fa6', 'iran_aligned': '#c0392b',
               'neutral': '#7e57c2', 'other': '#999999'}
CONCEPT_ORDER = ['diplomacy', 'nuclear_program', 'strike_claims', 'deterrence',
                 'military_action']

# ---- (a) edge construction, from the first sentence that shows the product --
example = None
with open(TAGGED) as fh:
    for line in fh:
        d = json.loads(line)
        pols = {CONCEPT_POLARITY[c] for c in d['concepts']}
        if (len(d['actors']) == 3 and len(d['concepts']) == 2
                and len(d['text']) < 190 and {'positive', 'negative'} <= pols):
            example = d
            break

if example is None:
    print('No example sentence matched the criteria; skipping figure (a).')
else:
    seen, alias_rows = set(), []
    for e in example['alias_log']:
        if e['resolved'] in example['actors'] and e['resolved'] not in seen:
            seen.add(e['resolved'])
            # a surface that is absent from the sentence text got there by the
            # coreference rewrite, which is worth showing explicitly
            alias_rows.append((e['surface'], e['resolved'],
                               e['surface'] not in example['text']))
    seen, con_rows = set(), []
    for e in example['concept_log']:
        if e['concept'] not in seen:
            seen.add(e['concept'])
            con_rows.append((e['trigger'], e['concept'], e['match_type'],
                             CONCEPT_POLARITY[e['concept']]))

    n_edges_ex = len(alias_rows) * len(con_rows)
    wrapped = example['text']
    if len(wrapped) > 95:
        cut = wrapped.rfind(' ', 0, 95)
        wrapped = wrapped[:cut] + '\n' + wrapped[cut + 1:]

    fig, ax = plt.subplots(figsize=(12.5, 6.2))
    ax.set_xlim(0, 100); ax.set_ylim(-11, 100); ax.axis('off')

    ax.text(2, 96, 'Stage 1   the sentence, after reference resolution',
            fontsize=9.5, weight='bold', color='#444444')
    ax.add_patch(FancyBboxPatch((2, 78), 96, 15,
                                boxstyle='round,pad=0,rounding_size=2.2',
                                fc='#f5f5f5', ec='#cccccc', lw=1.1, zorder=2))
    ax.text(50, 85.5, f'"{wrapped}"', ha='center', va='center', fontsize=11.5,
            style='italic')
    ax.text(98, 76.6, f"sentence {example['sentence_id']}, {example['source']}, "
                      f"{example['date']}, window {example['window']}",
            ha='right', va='top', fontsize=7.5, color='#777777')

    ax.text(2, 69, 'Stage 2   surface form to node', fontsize=9.5,
            weight='bold', color='#444444')
    ax.text(4, 66, 'actors (alias map)', fontsize=8.5, color='#777777',
            style='italic')
    for i, (surf, canon, via_coref) in enumerate(alias_rows):
        y = 60 - i * 6.5
        ax.text(4, y, f'"{surf}"', fontsize=9.5, va='center', color='#333333')
        ax.annotate('', xy=(30, y), xytext=(24, y),
                    arrowprops=dict(arrowstyle='->', color='#999999', lw=1))
        ax.text(31, y, canon, fontsize=9.5, va='center', weight='bold',
                family='monospace',
                color=ALIGN_COLOR.get(ACTOR_ALIGNMENT.get(canon), '#999999'))
        if via_coref:
            ax.text(31, y - 3.0, 'substituted for a pronoun by coreference',
                    fontsize=7.5, va='center', color='#888888', style='italic')

    ax.text(56, 66, 'concept triggers', fontsize=8.5, color='#777777',
            style='italic')
    for i, (trig, cluster, mtype, pol) in enumerate(con_rows):
        y = 60 - i * 6.5
        ax.text(56, y, f'"{trig}"', fontsize=9.5, va='center', color='#333333')
        ax.annotate('', xy=(74, y), xytext=(68, y),
                    arrowprops=dict(arrowstyle='->', color='#999999', lw=1))
        ax.text(75, y, cluster, fontsize=9.5, va='center', weight='bold',
                family='monospace', color=POL_COLOR_FIG[pol])
        ax.text(75, y - 3.0, f'{mtype} match, {pol}', fontsize=7.5,
                va='center', color='#888888', style='italic')

    ax.text(2, 38, f'Stage 3   the cartesian product: {len(alias_rows)} actors '
                   f' x  {len(con_rows)} concepts  =  {n_edges_ex} weighted edges',
            fontsize=9.5, weight='bold', color='#444444')
    axy = [28 - 10 * i for i in range(len(alias_rows))]
    cyy = [26 - 14 * i for i in range(len(con_rows))]
    AX, CX = 30, 64
    for (surf, canon, _), y in zip(alias_rows, axy):
        for (_, cluster, _, pol), cy in zip(con_rows, cyy):
            ax.plot([AX + 4, CX - 4], [y, cy], color=POL_COLOR_FIG[pol],
                    lw=2.0, alpha=.55, zorder=1, solid_capstyle='round')
    for (surf, canon, _), y in zip(alias_rows, axy):
        col = ALIGN_COLOR.get(ACTOR_ALIGNMENT.get(canon), '#999999')
        ax.scatter([AX], [y], s=430, c=col, zorder=3, edgecolors='white',
                   linewidths=1.4)
        ax.text(AX - 6, y, canon, ha='right', va='center', fontsize=9.5,
                weight='bold', family='monospace', color=col)
    for (_, cluster, _, pol), cy in zip(con_rows, cyy):
        ax.scatter([CX], [cy], s=560, marker='s', c=POL_COLOR_FIG[pol],
                   zorder=3, edgecolors='white', linewidths=1.4)
        ax.text(CX + 6, cy, cluster, ha='left', va='center', fontsize=9.5,
                weight='bold', family='monospace', color=POL_COLOR_FIG[pol])

    ax.text(50, -10, 'Each edge inherits the polarity of its concept and not '
            'the stance of the speaker, so this single\nsentence produces '
            'positive and negative edges while making one argument.',
            ha='center', va='bottom', fontsize=8.8, color='#555555')
    plt.tight_layout()
    out = DIR_NETFIG / 'edge_construction_example.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {out.name}   (sentence {example["sentence_id"]})')

# ---- (b) the bipartite graph, sparse window vs saturated window ------------
edge_w = {}
for w in present_windows:
    d = {}
    for line in open(EDGES_DIR / f'edges_{w}.jsonl'):
        r = json.loads(line)
        d[(r['actor'], r['concept'])] = r['weight_normalized']
    edge_w[w] = d

concepts_fig = [c for c in CONCEPT_ORDER if c in CONCEPT_POLARITY]
tot_w = {a: sum(edge_w[w].get((a, c), 0) for w in present_windows
                for c in concepts_fig) for a in ACTOR_WHITELIST}

# heaviest actors in the middle, lighter fanning outward: keeps the thick edges
# short so the drawing does not shear to one side
_by_w = sorted(ACTOR_WHITELIST, key=lambda a: -tot_w[a])
_n = len(_by_w); _c0 = _n // 2
_seq, _k = [_c0], 1
while len(_seq) < _n:
    if _c0 + _k < _n:
        _seq.append(_c0 + _k)
    if _c0 - _k >= 0:
        _seq.append(_c0 - _k)
    _k += 1
actors_fig = [None] * _n
for _a, _slot in zip(_by_w, _seq):
    actors_fig[_slot] = _a
apos = {a: i for i, a in enumerate(actors_fig)}
cxs = np.linspace(6, _n - 7, len(concepts_fig))

sparse = min(present_windows, key=lambda w: len(edge_w[w]))
dense = max(present_windows, key=lambda w: len(edge_w[w]))
n_poss = _n * len(concepts_fig)

fig, axes = plt.subplots(2, 1, figsize=(15, 8.4))
for ax, win in zip(axes, [sparse, dense]):
    W = edge_w[win]
    mx = max(W.values()) if W else 1
    for (a, c), v in sorted(W.items(), key=lambda kv: kv[1]):
        if c not in concepts_fig:
            continue
        ax.plot([apos[a], cxs[concepts_fig.index(c)]], [0, 1],
                color=POL_COLOR_FIG[CONCEPT_POLARITY[c]],
                lw=0.25 + 3.2 * (v / mx), alpha=0.10 + 0.55 * (v / mx),
                zorder=1, solid_capstyle='round')
    deg = [sum(W.get((a, c), 0) for c in concepts_fig) for a in actors_fig]
    mxd = max(deg) or 1
    ax.scatter(range(_n), [0] * _n, s=[9 + 120 * (d / mxd) for d in deg],
               c=[ALIGN_COLOR.get(ACTOR_ALIGNMENT.get(a), '#999999')
                  for a in actors_fig],
               zorder=3, edgecolors='white', linewidths=.5)
    ax.scatter(cxs, [1] * len(concepts_fig), s=260, marker='s', zorder=3,
               c=[POL_COLOR_FIG[CONCEPT_POLARITY[c]] for c in concepts_fig],
               edgecolors='white', linewidths=1.2)
    for x, c in zip(cxs, concepts_fig):
        ax.text(x, 1.09, c.replace('_', '\n'), ha='center', va='bottom',
                fontsize=8, weight='bold')
    for i, a in enumerate(actors_fig):
        ax.text(i, -0.07, a.replace('_', ' '), rotation=90, ha='center',
                va='top', fontsize=5.8,
                color='#222222' if deg[i] > 0 else '#bbbbbb')
    iso = sum(1 for d in deg if d == 0)
    ax.set_title(f'{win}   |   {len(W)} of {n_poss} edges realised, '
                 f'density {len(W)/n_poss:.3f}, {iso} isolated actors',
                 fontsize=10, loc='left')
    ax.set_xlim(-1.5, _n + .5); ax.set_ylim(-0.62, 1.30); ax.axis('off')
fig.suptitle('The bipartite actor-concept network: sparsest window against '
             'densest window', fontsize=12.5)
plt.tight_layout(rect=[0, 0, 1, 0.97])
out = DIR_NETFIG / 'bipartite_windows.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out.name}   ({sparse} vs {dense})')


## Step 12: Validation checkpoint

In [ ]:
outputs = [
    (DIR_GRAPH      / 'graph_level_metrics.csv',      'analysis/graph_level/'),
    (DIR_NODE       / 'node_level_centralities.csv',  'analysis/node_level/'),
    (DIR_NODE       / 'centrality_resolution.csv',    'analysis/node_level/'),
    (DIR_NODE       / 'node_katz_trajectories.png',   'analysis/node_level/'),
    (DIR_NODE       / 'coverage_diagnostics.png',     'analysis/node_level/'),
    (DIR_GROUP      / 'group_volume_share.csv',       'analysis/group_level/'),
    (DIR_GROUP      / 'group_concept_share.csv',      'analysis/group_level/'),
    (DIR_GROUP      / 'group_volume_share.png',       'analysis/group_level/'),
    (DIR_GROUP      / 'group_concept_share.png',      'analysis/group_level/'),
    (DIR_DYAD       / 'edge_weight_timeseries.png',   'analysis/dyad_level/'),
    (DIR_DYAD       / 'actor_concept_centrality.png', 'analysis/dyad_level/'),
    (DIR_PROJECTION / 'projection_metrics.csv',       'analysis/projection/'),
    (DIR_PROJECTION / 'projection_top_pairs.csv',     'analysis/projection/'),
    (DIR_POLARITY   / 'polarity_over_time.csv',       'analysis/polarity/'),
    (DIR_POLARITY   / 'polarity_over_time.png',       'analysis/polarity/'),
    (DIR_NETFIG / 'edge_construction_example.png', 'analysis/networks/'),
    (DIR_NETFIG / 'bipartite_windows.png',         'analysis/networks/'),
]

print('VALIDATION CHECKPOINT (07_analysis):')
print(f'  Windows analysed   : {len(present_windows)}  {present_windows}')
print(f'  Missing windows    : {len(missing_windows)}  {missing_windows}')
print(f'  Projections loaded : {len(projections)}')
print(f'  Eigenvector->Katz fallback windows: {len(fallback_windows)}/{len(present_windows)}')
print()
print('  Outputs by level of analysis:')
for p, where in outputs:
    print(f'    [{"OK" if p.exists() else "MISS"}]  {where}{p.name}')
print()
print('Weighted metrics carry the signal (dyad weights, Katz, group volume share,')
print('polarity volume share). Unweighted ones (degree/betweenness/closeness,')
print('polarity breadth) are reported as saturation diagnostics, not as rankings.')
print('Transitivity/reciprocity are NOT computed on the bipartite graph (0 / undefined')
print('by construction); reciprocity and triads live in 07b on the directed SVO net.')